In [ ]:
# Import the libraries

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Define settings
import os

pd.set_option("display.precision", 3)
data_dir_path = os.getcwd() + '/dataverse_files/'

In [ ]:
# Import data
import pyreadr

croissance_et_climat_decadaires = pyreadr.read_r(data_dir_path + "croissance_et_climat_decadaires.rds" ).popitem()[1]
valorisation_annuelle = pyreadr.read_r(data_dir_path + "valorisation_annuelle.rds" ).popitem()[1]

## Preparation of croissance et climat decadaires

In [ ]:
df = croissance_et_climat_decadaires
df.head()

In [ ]:
print(df.shape)
print(df.columns)
print(df.info())

In [ ]:
df.describe(include= "all")

In [ ]:
df["croissance"][
    (df["decade"] % 37 >= 1) &
    (df["decade"] % 37 <= 3) &
    (df["annee"] <= (df["annee"].max() - 5))
].mean()

Average of the growth over the first three decades of previous years

In [ ]:
df["croissance"][
    (df["decade"] == df["decade"].max()) &
    (df["annee"] <= (df["annee"].max() - 5))
].mean()

Average of the growth over the last decade of previous years

In [ ]:
df[["ucs", "safran", "sol", "gestion"]].corr()

## Feature extraction

Pedo-Climatic Units (PCU), result from the crossing of the climatic information (SAFRAN grid point) and soil information (UCS soil mapping units).

In [ ]:
df_unique_pcu = df.drop_duplicates(subset= ['ucs', 'safran'])[["ucs", "safran"]]
df_unique_pcu['pcu'] = df_unique_pcu.reset_index().index

df = pd.merge(df, df_unique_pcu, on= ['ucs','safran'])
df.insert(0, "pcu", df.pop('pcu'))
df.head()

In [ ]:
df[["ucs", "safran", "sol", "gestion", "pcu"]].corr()

In [ ]:
df_unique_pcu = df.drop_duplicates(subset= ['ucs','safran', 'sol', 'type_de_prairie', 'gestion'])[['ucs','safran', 'sol', 'type_de_prairie', 'gestion']]
df_unique_pcu['id'] = df_unique_pcu.reset_index().index

df = pd.merge(df, df_unique_pcu, on= ['ucs','safran','sol', 'type_de_prairie', 'gestion'])
df.insert(0, "id", df.pop('id'))
df.head()

In [ ]:
df[["id"]].drop_duplicates("id").count()["id"]

C'est le nombre de tuples (UCS, SAFRAN, SOL, prairie, gestion) diff?rents.

In [ ]:
df[["pcu"]].drop_duplicates("pcu").count()["pcu"]

C'est le nombre de tuples (UCS, SAFRAN) diff?rents.

In [ ]:
(df.groupby(["id", "annee"]).count()["decade"] == 37).all()

There is no more than one group of consecutive decades per year and per tuple (UCS, SAFRAN, SOL, prairie, gestion).

In [ ]:
(df.groupby(["id", "annee", "decade"]).count().max() == 1).all()

Moreover, there is only one entry per year and per decade for a fixed tuple (UCS, SAFRAN, SOL, prairie, gestion).

In [ ]:
df.groupby(["id", "decade"]).count()[["annee"]].describe().loc["min":"max"]

Finally, only the year will change over the whole simulated period, i.e. 30 years, for each decade and for any tuple (UCS, SAFRAN, SOL, prairie, gestion).

Therefore, all groups of decades have the same values for the tuple (UCS, SAFRAN, SOL, prairie, gestion) and only the year will change (over the whole simulated period, ie 30 years).

So (UCS, SAFRAN, SOL, prairie, gestion) is the identifier.

In [ ]:
identifier_columns = ["ucs", "safran", "sol", "type_de_prairie", "gestion"]

## Select years

In [ ]:
nb_annees = 5
annee_inf = df["annee"].max() - nb_annees
df = df.loc[(df["annee"] > annee_inf), :]

## Sort by ID, annee and then decade

In [ ]:
df.sort_values(by= identifier_columns+["annee", "decade"], inplace=True)

In [ ]:
identifier_columns.remove("type_de_prairie")

## Data cleansing

In [ ]:
(df.isna() | df.isnull()).sum()

In [ ]:
df.dropna(inplace= True)

In [ ]:
(df.isna() | df.isnull()).sum().sum() == 0

Now that we have removed the NaN and Null values, we need to check whether we have any missing decades among the years.

In [ ]:
(df.groupby(["id", "annee"]).count()["decade"] == 37).all()

All the (id, year) pairs have a number of decades equals to 37, which means all of them remain complete.

In [ ]:
df.drop_duplicates("id").count()["id"]

This is the number of tuples (UCS, SAFRAN, SOL, grassland, management), or IDs, that have full years.

In [ ]:
df.groupby(["id", "decade"]).count()[["annee"]].describe().loc["min":"max"]

## Correct the dataframe

In [ ]:
df["annee"] = df["annee"].astype("int64")
df["decade"] = df["decade"].astype("int64")

In [ ]:
print(df.info())

In [ ]:
croissance_et_climat_decadaires_preprocessed = df

# Preparation of annual valuations

In [ ]:
df = valorisation_annuelle
df.head()

## Feature selection

In [ ]:
valuable_columns = identifier_columns+["annee", "cumul_croissance"]
df = valorisation_annuelle.loc[:, valuable_columns]

In [ ]:
(df.isna() | df.isnull()).sum()

## Correct the dataframe

In [ ]:
df[["annee"]] = df[["annee"]].astype("int64")
df[["cumul_croissance"]] = df[["cumul_croissance"]].astype("float64")
df.head()

In [ ]:
valorisation_annuelle_preprocessed = df

# Concatenation

In [ ]:
X = croissance_et_climat_decadaires_preprocessed.merge(valorisation_annuelle_preprocessed, how= 'left', on= identifier_columns+["annee"])
X.iloc[35:39]

We can see that when a decade ends (passage from the 37th decade to the 1st), we change year (passage from 1985 to 1986) and we remain in the same tuple (UCS, SAFRAN, SOL, grassland, management), or ID.

In [ ]:
identifier_columns[3:3] += ["type_de_prairie"]

## Cleansing of abberations

In [ ]:
WrongCumul = X.groupby(identifier_columns+["annee"]).sum(numeric_only= True)["croissance"] / 100 != X.groupby(identifier_columns+["annee"]).last()["cumul_croissance"]
WrongCumul.sum()

This is the number of pairs (id, year) whose theoretical cumulative growth is not equal to the observed cumulative growth.

In [ ]:
X = X.merge(WrongCumul.reset_index(), how='left', on= identifier_columns+["annee"])
WrongIndexes = X[X[0] == True].index

X.drop(WrongIndexes, inplace= True)
X.drop(columns=0, inplace= True)
X.head()

## Split dataset

In [ ]:
safran_quantile_07 = X.drop_duplicates("safran")["safran"].quantile(0.7)
# Train set is 70% of the data set
train_index = X[X["safran"] <= safran_quantile_07].index
test_index = X[X["safran"] > safran_quantile_07].index

safran_test_median = X[X.index.isin(test_index)].drop_duplicates("safran")["safran"].median()
# Validation set is 15% of the data set
val_index = X[(X["safran"] <= safran_test_median) & (X["safran"] > safran_quantile_07)].index
# Test set is 15% of the data set
test_index = X[X["safran"] > safran_test_median].index

## Before and after the quantile make 100% of the safran values.
assert    X[X.index.isin(train_index)].drop_duplicates("safran")["safran"].count() \
       +  X[X.index.isin(val_index)].drop_duplicates("safran")["safran"].count() \
       +  X[X.index.isin(test_index)].drop_duplicates("safran")["safran"].count() \
       == X.drop_duplicates("safran")["safran"].count()

## Train set and test set make 100% of the dataset.
assert    len(X[X.index.isin(train_index)]) \
       +  len(X[X.index.isin(val_index)]) \
       +  len(X[X.index.isin(test_index)]) \
       == len(X)

In [ ]:
data_columns = ["Tmin", "Tmax", "Tmoy", "Rain", "RG", "im", "croissance", "cumul_croissance"]
X = X.loc[:, data_columns]
X.head()

In [ ]:
(X.isna() | X.isnull()).sum().sum() == 0

In [ ]:
Y = pd.DataFrame({'croissance':X.pop('croissance')})

In [ ]:
X_train = X[X.index.isin(train_index)]
X_val = X[X.index.isin(val_index)]
X_test = X[X.index.isin(test_index)]

Y_train = Y[Y.index.isin(train_index)]
Y_val = Y[Y.index.isin(val_index)]
Y_test = Y[Y.index.isin(test_index)]

In [ ]:
X_train

In [ ]:
Y_train